In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import pandas as pd

T_TARGET = 4 * 3600

datasets = {
    "Bursty": [
        {"id": "exp_20250625_014444", "name": "Openwhisk"},
        {'id': 'exp_20260923_212417', 'name': 'Unlock'},
        {'id': 'exp_20260923_092219', 'name': 'Process'},
        {"id": "exp_20260923_161047", "name": "Profiler"},
        {"id": "exp_20250625_071954", "name": "NMIG"},
    ],
    "Normal": [
        {"id": "exp_20250626_205804", "name": "Openwhisk"},
        {'id': 'exp_20260917_123621', 'name': 'Unlock'},
        {'id': 'exp_20260915_152838', 'name': 'Process'},
        {"id": "exp_20260910_221952", "name": "Profiler"},
        {"id": "exp_20250627_042542", "name": "NMIG"},
    ],
    "Similar": [
        {"id": "exp_20250625_162425", "name": "Openwhisk"},
        {'id': 'exp_20260922_131116', 'name': 'Unlock'},
        {'id': 'exp_20260922_173728', 'name': 'Process'},
        {"id": "exp_20260922_222414", "name": "Profiler"},
        {"id": "exp_20250625_113106", "name": "NMIG"},
    ],
}

all_results = []

for dataset_name, experiments in datasets.items():
    for exp in experiments:
        fn = pd.read_csv(f"../results/{exp['id']}/results_updated.csv")
        df = pd.read_csv(f"../results/{exp['id']}/gpu_usage.csv")

        initial_time_ms = fn['timestamp'].iloc[0]
        initial_dt = pd.to_datetime(initial_time_ms, unit='ms')

        df['second_dt'] = pd.to_datetime(df['Timestamp'], unit='ms').dt.floor('s')

        # Mean per container per second, then sum across containers
        df_avg = (df.groupby(['ContainerID', 'second_dt'])['GPU_Memory_MB']
                    .mean().reset_index())
        mem_per_second = (df_avg.groupby('second_dt')['GPU_Memory_MB']
                                .sum().reset_index(name='total_mem_mb'))

        mem_per_second['rel_s'] = (
            (mem_per_second['second_dt'] - initial_dt)
            .dt.total_seconds().astype(int)
        )
        mem_per_second = mem_per_second[mem_per_second['rel_s'] >= 0].reset_index(drop=True)

        # Uniform 4-hour grid, zeros where no data
        full = pd.DataFrame({'rel_s': range(T_TARGET + 1)})
        m = full.merge(mem_per_second[['rel_s', 'total_mem_mb']],
                       on='rel_s', how='left')
        m['total_mem_mb'] = m['total_mem_mb'].fillna(0)

        all_results.append({
            'Dataset':             dataset_name,
            'Configuration':       exp['name'],
            'GPU mem-time (GB·s)': round(m['total_mem_mb'].sum() / 1024.0, 2),
            'Mean resident (MB)':  round(m['total_mem_mb'].mean(), 2),
            'Peak (MB)':           round(m['total_mem_mb'].max(), 2),
        })

df_results = pd.DataFrame(all_results)

# Compute reduction % relative to Openwhisk within each dataset
def add_reductions(df):
    baseline = df[df['Configuration'] == 'Openwhisk'].iloc[0]
    df = df.copy()
    df['GPU mem reduction (%)'] = df['GPU mem-time (GB·s)'].apply(
        lambda x: round((1 - x / baseline['GPU mem-time (GB·s)']) * 100, 1)
    )
    df['Mean mem reduction (%)'] = df['Mean resident (MB)'].apply(
        lambda x: round((1 - x / baseline['Mean resident (MB)']) * 100, 1)
    )
    return df

df_results = df_results.groupby('Dataset', group_keys=False).apply(add_reductions)

# Print grouped by dataset
for dataset_name, group in df_results.groupby('Dataset'):
    print(f"\n{'='*70}")
    print(f"  {dataset_name}")
    print(f"{'='*70}")
    print(group.drop(columns='Dataset').to_string(index=False))


  Bursty
Configuration  GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
    Openwhisk             98869.66             7030.24    22910.0                    0.0                     0.0
       Unlock             39163.37             2784.76    10976.0                   60.4                    60.4
      Process             30161.84             2144.69    10976.0                   69.5                    69.5
     Profiler             30084.04             2139.16    10976.0                   69.6                    69.6
         NMIG               529.87               37.68    11071.0                   99.5                    99.5

  Normal
Configuration  GPU mem-time (GB·s)  Mean resident (MB)  Peak (MB)  GPU mem reduction (%)  Mean mem reduction (%)
    Openwhisk             44965.13             3197.30    13254.0                    0.0                     0.0
       Unlock             34425.18             2447.84     4067.0           

/tmp/ipykernel_456404/2579614962.py:81: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_results = df_results.groupby('Dataset', group_keys=False).apply(add_reductions)
